# Clone A Reconstruction

Cloning creates modified copies of a finalized reconstruction. In this tutorial, you will smooth dendrites, resample a reconstruction, and create several clones with different dendrite orientations.

## Workflow

```text
sample SWC file
      |
      v
smooth contracted dendrites
      |
      v
center and resample
      |
      v
find bifurcation points
      |
      v
create several clones
      |
      v
view all clones together
```

## Reading the clone commands

These examples use `swc modify` for smoothing and twisting, and `swc repair` for centering and resampling.

| Part | Meaning |
| --- | --- |
| `-m 2` | smooth sections with two iterations |
| `-t T` | stretch sections in their principal direction |
| `-n` | center the reconstruction at the origin |
| `-r 5` | resample with fixed spatial resolution of `5` um |
| `-g2` | find bifurcation points with degree `2` |
| `-w180` | rotate selected branches by random angles from `-180` to `180` degrees |
| `--seed` | make random changes repeatable |
| `clone?.swc` | match `clone1.swc`, `clone2.swc`, and `clone3.swc` |

## Command helper

Run this cell once before the tutorial commands. It creates `shell_cmd()`, a small notebook helper that runs terminal commands, shows their output, and displays images when a command creates one.

In [ ]:
import shutil
import subprocess
from pathlib import Path

from IPython.display import Image, display


def shell_cmd(command, image=None):
    print("Executed command:")
    print(command)
    bash_path = shutil.which("bash")
    result = subprocess.run(
        command,
        shell=True,
        executable=bash_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    output = result.stdout.rstrip()
    print("
Output:")
    print(output if output else "(empty)")

    if image is not None and Path(image).exists():
        display(Image(filename=image))


## Prepare working folders

Create temporary folders for input data and generated output files.

In [ ]:
shell_cmd("mkdir -pv data output")

## Copy a sample file

Use the cut NMO reconstruction for post-processing examples.

In [ ]:
shell_cmd("cp -v ../tests/data/pass_nmo_2_cut.swc data/")

## View contracted dendrites

Start by viewing basal dendrites before smoothing.

In [ ]:
shell_cmd(
    "f=data/pass_nmo_2_cut.swc; swc view $f -jxy -p3 --no-axes -o output/contracted_dendrites.png",
    image="output/contracted_dendrites.png",
)

## Smooth dendrites

Smoothing can be used as a post-processing step when dendrites are contracted.

In [ ]:
shell_cmd("f=data/pass_nmo_2_cut.swc; swc modify $f -p3 -m 2 -o output/fixsmooth.swc")
shell_cmd(
    "f=data/pass_nmo_2_cut.swc; swc view output/fixsmooth.swc $f -p3 -c shadow --no-axes -jxy -o output/smooth_compare.png",
    image="output/smooth_compare.png",
)

Smoothing methods: `-m M` is spatial low-pass filtering where `M` is number of iterations (`M=1,2,...`); `-t T` is section stretching in principal direction where `T` is relative stretching factor (`T>=1`).

## Center and resample

Another post-processing step is centering the finalized reconstruction at the origin with `-n` and resampling it with fixed spatial resolution using `-r R`, for example `5` um.

In [ ]:
shell_cmd("f=data/pass_nmo_2_cut.swc; swc repair $f -n -r 5 -o output/fixsample.swc")
shell_cmd(
    "f=data/pass_nmo_2_cut.swc; swc view output/fixsample.swc $f -p3 -jxy -c shadow --no-axes -o output/resample_compare.png",
    image="output/resample_compare.png",
)

## Find bifurcation points

Create clones of the final morphology that have the same total dendritic length and structure but different orientation of dendritic sections. First find bifurcation points in basal dendrites: point type `3`, degree `2`.

In [ ]:
shell_cmd("bifs=$(swc find output/fixsample.swc -p3 -g2); echo $bifs")
shell_cmd(
    "bifs=$(swc find output/fixsample.swc -p3 -g2); swc view output/fixsample.swc -m $bifs -p3 -jxy --no-axes -o output/bifurcations.png",
    image="output/bifurcations.png",
)

## Create clone morphologies

Rotate dendritic branches by a random angle from `-180` to `180` degrees at the bifurcation points.

In [ ]:
shell_cmd("bifs=$(swc find output/fixsample.swc -p3 -g2); swc modify output/fixsample.swc -i $bifs -w180 --seed 1 -o output/clone1.swc")
shell_cmd("bifs=$(swc find output/fixsample.swc -p3 -g2); swc modify output/fixsample.swc -i $bifs -w180 --seed 2 -o output/clone2.swc")
shell_cmd("bifs=$(swc find output/fixsample.swc -p3 -g2); swc modify output/fixsample.swc -i $bifs -w180 --seed 3 -o output/clone3.swc")

## View clones together

View the original resampled reconstruction and the three clones together.

In [ ]:
shell_cmd(
    "swc view output/fixsample.swc output/clone?.swc -p3 -jxy -c cells --no-axes -o output/clones.png",
    image="output/clones.png",
)

Branch twisting can be combined with random jittering (`-j J`) and scaling (`-s X Y Z`). Those changes affect the total dendritic length.